In [58]:
import json
from urllib.request import urlopen
from urllib.parse import urlencode

# 都市名 → 緯度経度の辞書
cities = {
    "Sapporo": {"lat": 43.06, "lon": 141.35},
    "Naha":    {"lat": 26.21, "lon": 127.68},
    "Kasiwa": {"lat":35.87,"lon":139.98}
}

past_days = 1       # 先頭(0番目)が昨日
forecast_days = 3   # 1番目=今日、2番目=明日
tomorrow = 2        # 明日の位置(添字)
daily_fields = "temperature_2m_max,temperature_2m_min,precipitation_probability_max"

results = {}

for city, place in cities.items():
    url = "https://api.open-meteo.com/v1/forecast?" + urlencode({
        "latitude": place["lat"], "longitude": place["lon"],
        "daily": daily_fields,
        "past_days": past_days, "forecast_days": forecast_days,
        "timezone": "auto",
    })
    with urlopen(url, timeout=10) as response:
        weather = json.load(response)

    d = weather["daily"]
    results[city] = {
        "date": d["time"][tomorrow],
        "max": d["temperature_2m_max"][tomorrow],
        "min": d["temperature_2m_min"][tomorrow],
        "rain": d["precipitation_probability_max"][tomorrow],
    }

results

{'Sapporo': {'date': '2026-11-13', 'max': 13.9, 'min': -0.5, 'rain': 0},
 'Naha': {'date': '2026-11-13', 'max': 26.5, 'min': 21.6, 'rain': 14},
 'Kasiwa': {'date': '2026-11-13', 'max': 19.4, 'min': 8.9, 'rain': 0}}

In [59]:
for city, r in results.items():
    rain = r["rain"]
    if rain is None:
        bar, rain_text = "データなし", "データなし"
    else:
        bar = "█" * (rain // 10)
        rain_text = f"{rain}%"
    print(f'{city:8} {r["date"]}  最高{r["max"]}℃ 最低{r["min"]}℃  降水確率 {bar} {rain_text}')

# --- ここからチームで決める ---
diff = results["Sapporo"]["max"] - results["Naha"]["max"]
print(f"\n最高気温の差: {diff:.1f}℃")

diff_line = 20   # 例: 何℃以上なら「服装を大きく変える」にする?
rain_line = 40   # 例: 何%以上なら「傘がいりそう」にする?
art = r"""
　　　　　～～～～～～～～、
　　　　/｜ｏ　〉┴〈_〉┴〈　0｜
　　　/　|　　｜◎｜◎　／）｜
　　　|ｏ　|　ｒ ヽ、_人 ／／ O｜
　　　｜｜　ヽ＼＿＿＿_フ　 |
　　　| Ｏ｜Ｏ ＼　　凵 凵　0 |　　　　/)
　　　｜0 |　　０ ＼⌒⌒V O｜　　／っ）
　　　/T　|　o　0　　｀ー_´ｏ　｜／／
　　 |_」|／|￣￣￣▽▼▽￣￣|
　　 | | |__｜＿＿＿　■＿＿＿|
　　｜| | =|＝＝＝＝▼＝＝＝|
"""

if abs(diff) >= diff_line:
    print("感想:最高気温の差、ジェットコースターかよｗｗｗ(ノಠ益ಠ)ノ")
else:
    print(art)
    print("感想:地球働け( ՞ਊ ՞)")

Sapporo  2026-11-13  最高13.9℃ 最低-0.5℃  降水確率  0%
Naha     2026-11-13  最高26.5℃ 最低21.6℃  降水確率 █ 14%
Kasiwa   2026-11-13  最高19.4℃ 最低8.9℃  降水確率  0%

最高気温の差: -12.6℃

　　　　　～～～～～～～～、
　　　　/｜ｏ　〉┴〈_〉┴〈　0｜
　　　/　|　　｜◎｜◎　／）｜
　　　|ｏ　|　ｒ ヽ、_人 ／／ O｜
　　　｜｜　ヽ＼＿＿＿_フ　 |
　　　| Ｏ｜Ｏ ＼　　凵 凵　0 |　　　　/)
　　　｜0 |　　０ ＼⌒⌒V O｜　　／っ）
　　　/T　|　o　0　　｀ー_´ｏ　｜／／
　　 |_」|／|￣￣￣▽▼▽￣￣|
　　 | | |__｜＿＿＿　■＿＿＿|
　　｜| | =|＝＝＝＝▼＝＝＝|

感想:地球働け( ՞ਊ ՞)
